# 04 · One-stage anchor-based detector: RetinaNet

**Model.** torchvision `retinanet_resnet50_fpn`, COCO-pretrained, same ResNet-50 + FPN backbone family as the Faster R-CNN in 02.
There is no proposal stage: two small conv heads run on every FPN level (P3–P7, strides 8–128) and, for **every anchor at every location**,
predict class scores (sigmoid per class) and a box offset. With 9 anchors per location that is ~170k anchors for a 1280×720 frame, almost all background.
**Focal loss** down-weights the easy background anchors, so the few positives are not drowned out.

**Protocol.** Identical to 02 (same processed frames, val frames, evaluator, input size, epochs, augmentation, seed), via the shared `common/engine.py`.
The learning rate follows torchvision's RetinaNet recipe scaled to batch 4 (0.01 for 16 images → 0.0025).

**Experiments** (each changes one thing relative to the baseline):

| Name | Change | Grounded in |
|---|---|---|
| `baseline` | torchvision defaults: anchors 32–813 px, ratios 0.5/1/2, IoU ≥ 0.5 positive / < 0.4 negative, focal α=0.25 γ=2 | reference |
| `data_anchors` | anchor sizes and aspect ratios chosen from the training boxes (section 1) | object sizes at the model's input scale |
| `iou_thresholds` | positive IoU ≥ 0.4, negative < 0.3 | tiny boxes rarely reach IoU 0.5 with any anchor (section 2) |
| `focal_gamma` | γ = 1.5 instead of 2 | less down-weighting: our positives are hard, small objects |
| `tuned` | combination of the changes that helped | final tuned one-stage model |

**Outputs**: `runs/retinanet/<experiment>/` with the same files as 02.

## Setup

In [ ]:
import functools
import json
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision.models.detection import RetinaNet_ResNet50_FPN_Weights, retinanet_resnet50_fpn
from torchvision.models.detection import retinanet as tv_retinanet
from torchvision.models.detection._utils import Matcher
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.ops import box_iou, sigmoid_focal_loss
from tqdm.auto import tqdm

ON_KAGGLE = Path("/kaggle/input").exists()
if ON_KAGGLE:
    SRC_DIR = Path("/kaggle/working/ai-hw2-multiobject-detection/src")
    DATA_ROOT = Path("/kaggle/input/COMPETITION_FOLDER")                    # TODO
    PROC_ROOT = Path("/kaggle/input/PROCESSED_DATASET/data_processed")      # TODO
    RUNS_ROOT = Path("/kaggle/working/runs")
else:
    SRC_DIR = Path.cwd()
    DATA_ROOT = Path("../multiple-object-video-detection-hw-2-ucu-ai-2026")
    PROC_ROOT = Path("../data_processed")
    RUNS_ROOT = Path("../runs")

sys.path.insert(0, str(SRC_DIR))
from common.analysis import GroundTruth, error_table, miss_and_confusion, show_frame
from common.data import CLASSES, CocoDetection, ImageList, collate, to_model_input
from common.engine import load_summaries, run_experiment
from common.evaluate import measure_fps
from common.submission import write_submission

GT_FILE = PROC_ROOT / "val" / "gt_eval_coco.json"
RUNS_DIR = RUNS_ROOT / "retinanet"
PATHS = dict(proc_root=PROC_ROOT, data_root=DATA_ROOT, gt_file=GT_FILE, runs_dir=RUNS_DIR)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_WANDB = False
print("Device:", DEVICE)

## Protocol

Same values as `PROTOCOL` in 02, except the learning rate (model-specific recipe).

In [ ]:
PROTOCOL = dict(
    min_size=720, max_size=1280,
    epochs=8,
    batch_size=4,
    lr=0.0025,                       # torchvision RetinaNet: 0.01 for 16 images, scaled to 4
    momentum=0.9, weight_decay=1e-4,
    warmup_iters=300,
    grad_clip=10.0,
    train_step=1,
    val_max_images=None,
    num_workers=2,
    seed=42,
    amp=True,
    pretrained=True,
)

def octave_sizes(base_sizes):
    """RetinaNet's 3 anchor scales per level: s, s*2^(1/3), s*2^(2/3)."""
    return tuple(tuple(int(round(s * 2 ** (k / 3))) for k in range(3)) for s in base_sizes)

BASE = dict(
    anchor_base=(32, 64, 128, 256, 512),     # one base size per level P3..P7 (torchvision default)
    aspect_ratios=(0.5, 1.0, 2.0),           # NOTE: torchvision defines these as height / width
    fg_iou_thresh=0.5, bg_iou_thresh=0.4,    # anchor IoU >= fg -> positive, < bg -> background, between -> ignored
    focal_alpha=0.25, focal_gamma=2.0,
    score_thresh=0.001, nms_thresh=0.5,
    detections_per_img=500, topk_candidates=1000,
)

## 1. Ground the anchors in the data

Anchor quality depends on the object sizes **the model sees**, i.e. after its internal resize (short side 720, long side ≤ 1280).
For every training box we compute its shape at that scale and the best IoU any anchor *shape* can reach with it (anchor and box centred on each other).
This is an upper bound: in the real model, anchor centres sit on a grid with the level's stride, so the IoU can only be lower (section 2 measures it).

Candidates:
- the default sizes (base 32 px on P3) vs. smaller bases (8–24 px);
- the default ratios (h/w = 0.5, 1, 2) vs. ratios from the data (median h/w of the lower, middle and upper third of the training boxes).

Each row is scored by the share of objects whose best anchor IoU reaches 0.5 (the positive threshold).

In [ ]:
train_coco = json.loads((PROC_ROOT / "train" / "annotations_coco.json").read_text())
img_size = {im["id"]: (im["width"], im["height"]) for im in train_coco["images"]}
wh = []
for a in train_coco["annotations"]:
    if a["iscrowd"]:
        continue
    W, H = img_size[a["image_id"]]
    s = min(PROTOCOL["min_size"] / min(W, H), PROTOCOL["max_size"] / max(W, H))  # model's resize factor
    wh.append((a["bbox"][2] * s, a["bbox"][3] * s))
wh = np.array(wh)
del train_coco
print(f"{len(wh)} training boxes; size sqrt(w*h) at model scale, percentiles:",
      {q: round(float(v), 1) for q, v in zip([5, 25, 50, 75, 95], np.percentile(np.sqrt(wh.prod(1)), [5, 25, 50, 75, 95]))})

# Data-driven aspect ratios (h/w): median of each third of the log-ratio distribution
log_r = np.sort(np.log(wh[:, 1] / wh[:, 0]))
DATA_RATIOS = tuple(round(float(np.exp(np.median(part))), 2) for part in np.array_split(log_r, 3))
print("Data-driven aspect ratios (h/w):", DATA_RATIOS)

def anchor_shapes(base_sizes, ratios):
    shapes = []
    for level in octave_sizes(base_sizes):
        for size in level:
            for r in ratios:              # same formula as torchvision: h/w = r, area = size^2
                shapes.append((size / math.sqrt(r), size * math.sqrt(r)))
    return np.array(shapes)

def shape_coverage(wh, shapes):
    """Best IoU between each box shape and any anchor shape (both centred at the same point)."""
    inter = np.minimum(wh[:, None, 0], shapes[None, :, 0]) * np.minimum(wh[:, None, 1], shapes[None, :, 1])
    union = wh.prod(1)[:, None] + shapes.prod(1)[None, :] - inter
    return (inter / union).max(1)

sample = wh[np.random.default_rng(0).choice(len(wh), min(len(wh), 50_000), replace=False)]
small = np.sqrt(sample.prod(1)) < 32
rows = []
for base0 in [8, 12, 16, 24, 32]:
    base = tuple(base0 * 2 ** i for i in range(5))
    for rname, ratios in [("default", (0.5, 1.0, 2.0)), ("data", DATA_RATIOS)]:
        best = shape_coverage(sample, anchor_shapes(base, ratios))
        rows.append({"P3 base": base0, "ratios": rname, "mean best IoU": best.mean(),
                     "IoU>=0.5": (best >= 0.5).mean(), "IoU>=0.4": (best >= 0.4).mean(),
                     "small objects IoU>=0.5": (best[small] >= 0.5).mean()})
coverage = pd.DataFrame(rows)
best_row = coverage.sort_values(["IoU>=0.5", "mean best IoU"], ascending=False).iloc[0]
DATA_BASE = tuple(int(best_row["P3 base"]) * 2 ** i for i in range(5))
DATA_ANCHOR_RATIOS = (0.5, 1.0, 2.0) if best_row["ratios"] == "default" else DATA_RATIOS
print(f"Selected: P3 base {DATA_BASE[0]} px (levels {DATA_BASE}), ratios {DATA_ANCHOR_RATIOS}")
coverage.style.format({c: "{:.3f}" for c in coverage.columns[2:]}).highlight_max(subset=["IoU>=0.5"])

In [ ]:
sizes = np.sqrt(wh.prod(1))
fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(sizes, bins=np.logspace(0, np.log10(sizes.max()), 80), color="grey", alpha=0.6)
for base, color, label in [(BASE["anchor_base"], "C3", "default anchors"), (DATA_BASE, "C2", "data anchors")]:
    for i, s in enumerate(np.ravel(octave_sizes(base))):
        ax.axvline(s, color=color, lw=1, alpha=0.8, label=label if i == 0 else None)
ax.set(xscale="log", xlabel="object size sqrt(w*h) at model input scale, px", ylabel="number of training boxes",
       title="Object sizes vs anchor sizes")
ax.legend(); plt.show()

## Experiments

`data_anchors` uses the sizes and ratios selected above; everything else is a single change relative to `baseline`.

In [ ]:
EXPERIMENTS = {
    "baseline": {},
    "data_anchors": {"anchor_base": DATA_BASE, "aspect_ratios": DATA_ANCHOR_RATIOS},
    "iou_thresholds": {"fg_iou_thresh": 0.4, "bg_iou_thresh": 0.3},
    "focal_gamma": {"focal_gamma": 1.5},
    # Fill in after comparing the runs above: combine the changes that improved mAP.
    "tuned": {},
}

## Model

- The anchor generator keeps 3 scales × 3 ratios = 9 anchors per location, so the pretrained box-regression head still loads.
- Only the last classification layer is replaced (8 classes + an unused background slot, so labels stay 1..8 as in 02), with RetinaNet's prior-probability bias init.
- Focal loss α/γ are hard-coded in torchvision; we swap in a version with our values by replacing `torchvision.models.detection.retinanet.sigmoid_focal_loss`.

In [ ]:
def set_focal(alpha, gamma):
    tv_retinanet.sigmoid_focal_loss = functools.partial(sigmoid_focal_loss, alpha=alpha, gamma=gamma)

def build_model(cfg, pretrained=None):
    pretrained = PROTOCOL["pretrained"] if pretrained is None else pretrained
    set_focal(cfg["focal_alpha"], cfg["focal_gamma"])
    anchors = AnchorGenerator(octave_sizes(cfg["anchor_base"]), (tuple(cfg["aspect_ratios"]),) * 5)
    model = retinanet_resnet50_fpn(
        weights=RetinaNet_ResNet50_FPN_Weights.COCO_V1 if pretrained else None,
        weights_backbone=None,
        min_size=PROTOCOL["min_size"], max_size=PROTOCOL["max_size"],
        anchor_generator=anchors,
        fg_iou_thresh=cfg["fg_iou_thresh"], bg_iou_thresh=cfg["bg_iou_thresh"],
        score_thresh=cfg["score_thresh"], nms_thresh=cfg["nms_thresh"],
        detections_per_img=cfg["detections_per_img"], topk_candidates=cfg["topk_candidates"],
    )
    head = model.head.classification_head
    num_classes = len(CLASSES) + 1
    head.cls_logits = nn.Conv2d(256, head.num_anchors * num_classes, kernel_size=3, padding=1)
    nn.init.normal_(head.cls_logits.weight, std=0.01)
    nn.init.constant_(head.cls_logits.bias, -math.log((1 - 0.01) / 0.01))
    head.num_classes = num_classes
    return model

def load_trained(name):
    model = build_model({**BASE, **EXPERIMENTS[name]}, pretrained=False)
    model.load_state_dict(torch.load(RUNS_DIR / name / "best.pt", map_location="cpu"))
    return model.to(DEVICE).eval()

## 2. What the missing proposal stage means: anchor assignment statistics

No training needed: for 100 training frames we generate every anchor, match it to the ground truth with each experiment's IoU thresholds and count:
- **anchors per image** and how many are positive / background / ignored: the foreground–background imbalance focal loss has to handle;
- **objects with no anchor at IoU ≥ positive threshold**: they still get their single best anchor (torchvision's `allow_low_quality_matches`), but only a poor one;
- **positive anchors per object**, by object size.

A two-stage detector avoids most of this: its RPN keeps only ~1000 proposals per image, and the RoI head samples a fixed 25% positives from them.

In [ ]:
@torch.no_grad()
def anchor_stats(cfg, n_images=100):
    model = build_model(cfg, pretrained=False).eval()
    ds = CocoDetection(PROC_ROOT / "train" / "annotations_coco.json", PROC_ROOT / "train")
    matcher = Matcher(cfg["fg_iou_thresh"], cfg["bg_iou_thresh"], allow_low_quality_matches=True)
    per_img, per_obj = [], []
    for i in np.linspace(0, len(ds) - 1, n_images).astype(int):
        img, tgt = ds[i]
        if len(tgt["boxes"]) == 0:
            continue
        images, targets = model.transform(to_model_input([img], torch.device("cpu")), [tgt])
        h, w = images.tensors.shape[-2:]
        dims = [(math.ceil(h / s), math.ceil(w / s)) for s in (8, 16, 32, 64, 128)]
        feats = [torch.empty(1, 256, fh, fw) for fh, fw in dims]   # only the shapes matter
        anchors = model.anchor_generator(images, feats)[0]
        gt = targets[0]["boxes"]
        iou = box_iou(gt, anchors)
        m = matcher(iou)
        per_img.append({"anchors": len(anchors), "positive": int((m >= 0).sum()),
                        "background": int((m == Matcher.BELOW_LOW_THRESHOLD).sum()),
                        "ignored": int((m == Matcher.BETWEEN_THRESHOLDS).sum())})
        pos_per_gt = torch.bincount(m[m >= 0], minlength=len(gt))
        best = iou.max(1).values
        for b, n, bi in zip(gt.numpy(), pos_per_gt.numpy(), best.numpy()):
            per_obj.append({"size": "small" if (b[2] - b[0]) * (b[3] - b[1]) < 32**2 else "medium+",
                            "positive anchors": n, "no anchor >= fg": bi < cfg["fg_iou_thresh"]})
    return pd.DataFrame(per_img), pd.DataFrame(per_obj)

rows = {}
for name in ["baseline", "data_anchors", "iou_thresholds"]:
    img_df, obj_df = anchor_stats({**BASE, **EXPERIMENTS[name]})
    rows[name] = {"anchors / image": img_df.anchors.mean(), "positive / image": img_df.positive.mean(),
                  "background : positive": img_df.background.sum() / max(img_df.positive.sum(), 1),
                  "objects with no anchor >= fg": obj_df["no anchor >= fg"].mean(),
                  "small objects with no anchor >= fg": obj_df.loc[obj_df["size"] == "small", "no anchor >= fg"].mean(),
                  "positive anchors / small object": obj_df.loc[obj_df["size"] == "small", "positive anchors"].mean(),
                  "positive anchors / medium+ object": obj_df.loc[obj_df["size"] == "medium+", "positive anchors"].mean()}
pd.DataFrame(rows).round(3)

**Observations:** _fill in: how extreme the imbalance is, and how the anchor and threshold changes affect small objects._

## Run experiments

Same cost as Faster R-CNN (~40 min per run on Person A's GPU). Finished runs are saved, so a kernel restart loses nothing.

In [ ]:
RUN = ["baseline"]          # then ["data_anchors", "iou_thresholds", "focal_gamma"], later ["tuned"]
for name in RUN:
    run_experiment(name, {**BASE, **EXPERIMENTS[name]}, build_model, PROTOCOL, PATHS, DEVICE,
                   family="retinanet", use_wandb=USE_WANDB)

## Compare: RetinaNet experiments vs. the tuned two-stage detector

The Faster R-CNN row is its `tuned` run if it exists, otherwise its best-scoring run.

In [ ]:
retina = load_summaries(RUNS_DIR)
frcnn = load_summaries(RUNS_ROOT / "faster_rcnn")
rows = {f"retinanet/{n}": r["best"] for n, r in retina.items()}
if frcnn:
    fr_name = "tuned" if "tuned" in frcnn else max(frcnn, key=lambda n: frcnn[n]["best"]["mAP"])
    rows[f"faster_rcnn/{fr_name}"] = frcnn[fr_name]["best"]
cols = ["epoch", "mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR"]
table = pd.DataFrame({n: {c: b[c] for c in cols} for n, b in rows.items()}).T
table.style.format({c: "{:.4f}" for c in cols[1:]}).highlight_max(axis=0, subset=cols[1:])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for label, runs, root in [("retinanet", retina, RUNS_DIR), ("faster_rcnn", frcnn, RUNS_ROOT / "faster_rcnn")]:
    for name in runs:
        if label == "faster_rcnn" and name != fr_name:
            continue
        h = pd.DataFrame(json.loads((root / name / "history.json").read_text()))
        style = "--" if label == "faster_rcnn" else "-"
        for ax, col in zip(axes, ["mAP", "AP_small", "AP_large"]):
            ax.plot(h.epoch, h[col], style, marker="o", label=f"{label}/{name}")
for ax, t in zip(axes, ["val mAP@[.5:.95]", "val AP_small", "val AP_large"]):
    ax.set(xlabel="epoch", ylabel=t, title=t)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
per_class = pd.DataFrame({n: {c: b[f"AP_{c}"] for c in CLASSES} for n, b in rows.items()})
per_class.plot.bar(figsize=(13, 4), ylabel="AP@[.5:.95]", title="Per-class AP")
plt.show()
per_class.round(4)

## Inference speed (batch 1, 50 val frames)

In [ ]:
val_ds = CocoDetection(GT_FILE, DATA_ROOT)
speed_images = [val_ds[i][0] for i in range(0, len(val_ds), max(1, len(val_ds) // 50))][:50]
fps = {f"retinanet/{n}": measure_fps(load_trained(n), speed_images, DEVICE) for n in retina}
table["FPS"] = pd.Series(fps)
table.round(4)

## Failure analysis

Same matching rules as 02 (`common/analysis.py`): green = correct, red = missed, yellow = false positive, magenta = wrong class.

In [ ]:
ANALYSE = "baseline"     # switch to the tuned run once it exists
SCORE_THR = 0.3
gt = GroundTruth(GT_FILE)
dets = json.loads((RUNS_DIR / ANALYSE / "val_dets.json").read_text())
errors = error_table(gt, dets, SCORE_THR)
print("Totals over val:", errors[["n_gt", "missed", "fp", "wrong_class"]].sum().to_dict())
errors.groupby("video")[["n_gt", "missed", "fp", "wrong_class"]].sum()

In [ ]:
for col, title in [("missed", "Most missed objects"), ("fp", "Most false positives"), ("wrong_class", "Most class confusions")]:
    worst = errors.sort_values(col, ascending=False).iloc[0]
    show_frame(gt, dets, int(worst.image_id), DATA_ROOT, SCORE_THR, f"{title}: {int(worst[col])} of {int(worst.n_gt)} objects")

In [ ]:
miss, confusion = miss_and_confusion(gt, dets, SCORE_THR)
print("Miss rate by class and size:"); display(miss.round(3))
if len(confusion):
    print("Class confusions (true -> predicted):"); display(confusion)

**Discussion (fill in):**
- *Does removing the proposal stage cause issues?* Use section 2 (imbalance, objects without a good anchor) and the comparison with Faster R-CNN by object size.
- *How can they be addressed?* E.g. focal loss (already used), anchors matched to the data, lower IoU thresholds or adaptive assignment (ATSS), a stride-4 level for tiny objects.
- *Initial vs tuned RetinaNet vs tuned Faster R-CNN*: which changes helped, on which sizes and classes.

## Test submission

In [ ]:
SUBMIT = "baseline"

@torch.no_grad()
def predict_test(model):
    test = pd.read_csv(DATA_ROOT / "test_images.csv")
    dl = torch.utils.data.DataLoader(ImageList([DATA_ROOT / f for f in test["file_name"]], test["image_id"]),
                                     batch_size=2, num_workers=PROTOCOL["num_workers"], collate_fn=collate)
    preds = {}
    for images, keys in tqdm(dl, desc="test"):
        for out, key in zip(model(to_model_input(images, DEVICE)), keys):
            keep = out["labels"] >= 1
            b = out["boxes"][keep].cpu().numpy()
            preds[key] = [(int(c) - 1, float(s), x1, y1, x2 - x1, y2 - y1) for (x1, y1, x2, y2), s, c
                          in zip(b, out["scores"][keep].cpu().numpy(), out["labels"][keep].cpu().numpy())]
    return preds

sub_path = RUNS_DIR / SUBMIT / "submission.csv"
sub = write_submission(predict_test(load_trained(SUBMIT)), DATA_ROOT / "test_images.csv", sub_path)
print(f"Wrote {sub_path}: {len(sub)} rows, {sub.PredictionString.str.len().gt(0).sum()} with detections")